# Functional neurotransmitter fingerprinting

In this tutorial, you will explore the functional neurotransmitter fingerprinting (FNTF) functionalities of Lacuna using the CLI.

**What you'll learn**:

- Fetch the neurotransmitter PET atlas and a functional connectome
- Prepare the atlas
- Compute NT scores weighted by functional connectivity
- Use ACE-enriched atlases for enhanced mapping

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-petersen/lacuna/blob/main/docs/tutorials/08-functional-neurotransmitter-fingerprinting.ipynb)

## Colab

Note: Colab provides limited computational resources. While these tutorials are designed to operate within those constraints, some Lacuna functionality cannot be fully demonstrated in this environment and requires access to higher-performance computing infrastructure.

Ignore this if you run this notebook locally.

## Setup

In [ ]:
# Install Lacuna from GitHub
!pip install git+https://github.com/m-petersen/lacuna

Get the tutorial data.

In [ ]:
# Get tutorial data
!lacuna tutorial /tmp/tutorial_bids --force

## Fetch data

FNTF requires two data sources:

1. **Neurotransmitter PET atlas** — curated representative PET receptor / transporter density maps from [NiSpace-data](https://github.com/LeonDLotter/NiSpace-data), pinned to a specific commit and verified by SHA-256.
2. **Functional connectome** — Resting-state fMRI data from a normative cohort (e.g., [GSP1000](https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/ILXIKS)).

For the GSP1000 download you need an API token. Register on [Harvard Dataverse](https://dataverse.harvard.edu/), click on your username and then API Token.

Here, we fetch the test version via `--test-mode` to keep the tutorial lightweight.

*Note on the cerebellum:* in most PET maps the cerebellum (or a cerebellar grey-matter sub-region) is used as the kinetic-modelling reference — non-specific tracer binding there is divided out of every voxel, so the cerebellum's own values become uninterpretable and are masked out. Expect zero / NaN cerebellar values in the downloaded maps.

In [ ]:
# Fetch NT atlas
!lacuna fetch ntatlas \
    --output-dir /tmp/ntatlas_data

In [ ]:
# Fetch functional connectome (test mode)
!lacuna fetch gsp1000 \
    --output-dir /tmp/gsp1000_data \
    --api-key $DATAVERSE_API_KEY \
    --test-mode \
    --skip-checksum \
    --no-keep-original

## Analysis

Functional neurotransmitter fingerprinting combines functional connectivity with neurotransmitter information. It:

1. Computes the functional connectivity map of the lesion using the normative fMRI connectome
2. Weights NT atlas values by the functional connectivity at each voxel
3. Scores each neurotransmitter target based on connectivity-weighted NT values

This answers: **what NT systems are functionally connected to the lesion?**

A high score for a given target indicates that brain regions functionally connected to the lesion are rich in that neurotransmitter, suggesting potential remote neurochemical effects.

Run the analysis.

In [ ]:
!lacuna run fntf \
    /tmp/tutorial_bids/ \
    /tmp/outputs_fntf/ \
    --connectome-name GSP1000 \
    --participant-label 01 \
    --mask-space MNI152NLin6Asym \
    --ntatlas-dir /tmp/ntatlas_data

List the outputs.

In [ ]:
!ls /tmp/outputs_fntf/sub-01/ses-01/anat/

## ACE-enriched mode

For enriched scoring you can swap `--ntatlas-dir` for `--ace-cache-dir`. ACE (Atlas Connectivity Enrichment) precomputes connectivity-informed NT timeseries; FNTF then uses them for temporal-correlation scoring instead of the static atlas × z-map dot product.

```bash
# Prepare the ACE cache once
lacuna prepare ace --connectome-name GSP1000

# Run with the ACE cache instead of the static atlas
lacuna run fntf /bids/ /output/ \
    --connectome-name GSP1000 \
    --ace-cache-dir /path/to/ace
```

Note: ACE preparation is expensive (per-subject GLM on the connectome) — only do it once per connectome.

## Radar plot of the fingerprint

A radar plot is a natural way to visualise an NT fingerprint — each axis is a target, the radial extent is the score. We render one plot for sub-01 from the TSV produced above; the same code generalises to a loop over subjects.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

tsv = next(Path("/tmp/outputs_fntf/sub-01/ses-01/anat").glob("*fntf*profilestats.tsv"))
df = pd.read_csv(tsv, sep="\t")

targets = df["target"].tolist()
values = df["value"].to_numpy()
angles = np.linspace(0, 2 * np.pi, len(targets), endpoint=False)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(projection="polar"))
ax.plot(np.append(angles, angles[0]), np.append(values, values[0]),
        color="steelblue", linewidth=1.5)
ax.fill(np.append(angles, angles[0]), np.append(values, values[0]),
        color="steelblue", alpha=0.25)
ax.set_xticks(angles)
ax.set_xticklabels(targets, fontsize=9)
ax.set_title("sub-01", pad=20)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)
plt.tight_layout()
plt.show()
